# 21 · «Серьёзная тревога» с окном подтверждения 24 ЧАСА (против 6ч и 30м)

Продолжение §17: 30 мин оказалось хуже 6ч (PR-AUC 24ч 0.42 → 0.12). Проверяем
противоположную сторону — **≥2 дымовых каналов объекта в пределах 24 часов**
(эпизоды с разрывом ≤24ч) + ручной ИПР. Сравниваем 30м / 6ч / 24ч на том же
holdout (дефолты 1000/0.03/10).

Сырые события уже собраны (ноутбук 20) — читаем `fire_raw_events_30m.parquet`,
журналы не перечитываем.


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import survival_metrics, pr_at_horizon, fit_discrete_hazard


DATASET = du.find_dataset_dir()
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9
GAP_NS = 24 * 3600 * 10 ** 9          # 24 часа
EV = pd.read_parquet(DATASET / "fire_raw_events_30m.parquet")
print("сырых событий:", len(EV), "| дым:", int((EV["тип"] == "дым").sum()),
      "| ручной:", int((EV["тип"] == "ручной").sum()))


сырых событий: 164477 | дым: 164288 | ручной: 189


In [2]:
def detect(ev_ob, gap_ns):
    ev_ob = ev_ob.sort_values("t").reset_index(drop=True)
    t = ev_ob["t"].to_numpy()
    gid = np.cumsum(np.concatenate([[True], t[1:] - t[:-1] > gap_ns])) - 1
    ev2 = ev_ob.copy(); ev2["gid"] = gid
    cnt = ev2[ev2["тип"] == "дым"].groupby("gid")["ид"].nunique().rename("n_dym")
    agg = ev2.groupby("gid").agg(t0=("t", "min"), t1=("t", "max"),
                                 n_manual=("тип", lambda s: int((s == "ручной").sum())))
    agg = agg.join(cnt).fillna({"n_dym": 0})
    agg["serious"] = (agg["n_dym"] >= 2) | (agg["n_manual"] > 0)
    agg["бакет"] = (agg["t0"] // hour_ns).astype(np.int64)
    return agg[agg["serious"]]

EPISODES = []
for об, grp in EV.groupby("об"):
    d = detect(grp, GAP_NS)
    if len(d):
        d = d.reset_index().rename(columns={"index": "gid"})
        d["об"] = об
        EPISODES.append(d)
EP = pd.concat(EPISODES, ignore_index=True) if EPISODES else pd.DataFrame()
EP.to_parquet(DATASET / "fire_episodes_24h.parquet")
print("серьёзных эпизодов (24ч):", len(EP),
      "| ручных:", int((EP["n_manual"] > 0).sum()),
      "| дымовых (>=2 ИПР):", int((EP["n_manual"] == 0).sum()))


серьёзных эпизодов (24ч): 1616 | ручных: 39 | дымовых (>=2 ИПР): 1577


In [3]:
# Колонка серьёзных_24ч на панели (тот же способ проекции, что в §17)
ep = pd.read_parquet(DATASET / "fire_episodes_24h.parquet")
ep["об"] = ep["об"].astype(str)
start_map = ep[["об", "бакет"]].drop_duplicates().rename(columns={"об": "ид_объект"})

panel = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
panel = panel.merge(start_map, on=["ид_объект", "бакет"], how="left", indicator="m")
panel["серьёзных_24ч"] = (panel["m"] == "both").astype(np.int8)
panel = panel.drop(columns=["m"])
print("база события 24ч на панели: {:.3f}%".format(panel["серьёзных_24ч"].mean() * 100))
print("для сравнения: 6ч = 8.522%, 30м = 11.801%")


база события 24ч на панели: 7.179%
для сравнения: 6ч = 8.522%, 30м = 11.801%


In [4]:
# Субъекты TTE и stats
panel = tte.refit_z_train_only(panel)
panel = tte.add_series_context(panel, event_col="серьёзных_24ч")
subjects = tte.add_tte_labels(panel)
subjects["ид_объект_code"] = subjects["ид_объект"].astype("category").cat.codes
subjects["тип_датчика_code"] = subjects["тип_датчика"].astype("category").cat.codes
tr, va, ho = tte.split_subjects(subjects)
ev = tr["event_flag"].fillna(0).to_numpy(); od = tr["obs_days"].to_numpy()
print("train база 24ч: <=1д {:.3f}% | <=7д {:.3f}% | <=30д {:.3f}%".format(
    ((ev == 1) & (od <= 1.01)).mean() * 100,
    ((ev == 1) & (od <= 7.01)).mean() * 100,
    ((ev == 1) & (od <= 30.01)).mean() * 100))
print("серий-событий 24ч в train (D<=1д):", int(((ev == 1) & (od <= 1.01)).sum()))


train база 24ч: <=1д 1.584% | <=7д 8.229% | <=30д 20.309%
серий-событий 24ч в train (D<=1д): 13496


In [5]:
def run(X_cols):
    trs = tte.sample_subjects(tr, 15000, 10000, 10000, seed=42)
    vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
    hos = tte.sample_holdout(ho, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(trs, X_cols, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(vas, X_cols, tte.HORIZON_BUCKETS)
    m = fit_discrete_hazard(X_tr, y_tr, X_va, y_va, iters=1000, lr=0.03,
                            depth=10, seed=42)
    surv_tr = tte.make_surv_struct(trs["event_flag"], trs["obs_days"])
    surv_ho = tte.make_surv_struct(hos["event_flag"], hos["obs_days"])
    S = np.cumprod(1 - m.predict_proba(
        tte.expand_person_time(hos, X_cols, tte.HORIZON_BUCKETS, fixed_horizon=True)[0]
        )[:, 1].reshape(-1, tte.HORIZON_BUCKETS), axis=1)
    risk = 1 - S[:, -1]
    met = survival_metrics(surv_tr, surv_ho, risk, S, tte.EVAL_TIMES)
    y24 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    return met, pr

X_cols = tte.subject_features(tr)
met24, pr24 = run(X_cols)
print(f"[серьёзная 24ч] Uno-C={met24['uno_c']:.3f} | IBS={met24['ibs']:.3f} "
      f"| P(24ч) PR-AUC={pr24['pr_auc']:.3f} (база {pr24['base']*100:.2f}%)")
print()
print("Сводка по окнам (тот же holdout, дефолты 1000/0.03/10):")
print("  30м: Uno-C 0.696, PR-AUC(24ч) 0.120, база 3.24%  (§17)")
print("  6ч : Uno-C 0.792, PR-AUC(24ч) 0.421, база 1.53%  (§9)")
print("  24ч: выше.")


0:	learn: 0.5997891	test: 0.6003943	best: 0.6003943 (0)	total: 1.22s	remaining: 20m 20s


100:	learn: 0.0069801	test: 0.0092064	best: 0.0091354 (89)	total: 1m 50s	remaining: 16m 25s


200:	learn: 0.0052390	test: 0.0091266	best: 0.0090839 (164)	total: 3m 40s	remaining: 14m 36s


300:	learn: 0.0047529	test: 0.0092777	best: 0.0090839 (164)	total: 5m 30s	remaining: 12m 47s


400:	learn: 0.0044385	test: 0.0094641	best: 0.0090839 (164)	total: 7m 19s	remaining: 10m 55s


Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.009083869041
bestIteration = 164

Shrink model to first 165 iterations.


[серьёзная 24ч] Uno-C=0.637 | IBS=0.062 | P(24ч) PR-AUC=0.196 (база 2.06%)

Сводка по окнам (тот же holdout, дефолты 1000/0.03/10):
  30м: Uno-C 0.696, PR-AUC(24ч) 0.120, база 3.24%  (§17)
  6ч : Uno-C 0.792, PR-AUC(24ч) 0.421, база 1.53%  (§9)
  24ч: выше.
